# Checkpoint 2: Writing Null/Alternative Hypotheses
### Dataset: cars.csv

https://www.kaggle.com/datasets/sehriyarmemmedli/turboaz-cars-project

"Automatic cars are more expensive" is not the same as "adding an automatic gearbox makes the price go up." My hypotheses are only about association/difference, not causation.

## 1. Loading the df_clean

In [56]:
import pandas as pd
import numpy as np
from scipy.stats import skew

pd.set_option('display.max_columns', 40)


In [57]:
cols = [
    "id_x", "car_rel_url_x", "datetime_scrape", "price_x", "currency_x", "city",
    "production_year", "engine_displacement_num", "kilometrage_num", "Marka", "Model",
    "Sürətlər qutusu", "Vəziyyəti", "Ötürücü", "Ban növü", "views"
]

In [58]:
df = pd.read_csv("cars.csv", usecols = cols, parse_dates = ["datetime_scrape"])
df

,id_x,car_rel_url_x,datetime_scrape,price_x,currency_x,city,production_year,engine_displacement_num,kilometrage_num,views,Ban növü,Marka,Model,Sürətlər qutusu,Vəziyyəti,Ötürücü
0,3c234145-d57a-4ad6-9448-d43810fc3392,/autos/8748840-hyundai-i30,2024-09-13 20:32:19.751157+00,15000.0,AZN,bakı,2008,1.6,270000,492,"Hetçbek, 5 qapı",Hyundai,i30,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Ön
1,c74ea36f-6be1-4de4-926d-e117197dcf00,/autos/8475807-lada-vaz-niva-travel,2024-09-13 20:32:19.751157+00,23700.0,AZN,bakı,2024,1.7,0,60189,"Offroader / SUV, 5 qapı",LADA (VAZ),Niva Travel,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
2,9cefceb0-024d-4581-a869-a3c2c68a9f95,/autos/8739686-toyota-land-cruiser,2024-09-13 20:32:19.751157+00,35600.0,$,bakı,2011,4.0,164750,2473,"Offroader / SUV, 5 qapı",Toyota,Land Cruiser,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam
3,459cc337-fb63-48de-9694-41554923d311,/autos/8712597-hyundai-elantra,2024-09-13 20:32:19.751157+00,26700.0,AZN,bakı,2018,2.0,126000,3727,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
4,6c5ee8d8-1c6f-4fad-a694-957a4c43c25d,/autos/8674773-toyota-prius,2024-09-13 20:32:19.751157+00,10500.0,AZN,bakı,2007,1.5,354000,446,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653716,16caa803-a546-455b-81ff-bc20868c2136,/autos/9081944-toyota-prius,2025-01-05 20:15:21.051803,10800.0,AZN,bakı,2008,1.5,320000,210,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
653717,2a26a6c1-8914-4b68-abb4-1fbd12ced7bc,/autos/9081939-uaz-hunter,2025-01-05 20:15:21.051803,9500.0,AZN,göygöl,2011,2.9,155000,1195,"Offroader / SUV, 5 qapı",UAZ,Hunter,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
653718,feb75615-0131-4d6f-b009-cc02faea4e01,/autos/9055034-hyundai-elantra,2025-01-05 20:15:21.051803,25400.0,AZN,bakı,2018,2.0,77926,1120,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
653719,e5f8e957-5543-42be-b30f-78723ce551f9,/autos/9065903-jeep-grand-cherokee,2025-01-05 20:15:21.051803,10600.0,AZN,kürdəmir,1999,4.7,250000,1320,"Offroader / SUV, 5 qapı",Jeep,Grand Cherokee,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam


In [59]:
df_dedup = df.sort_values("datetime_scrape").drop_duplicates(subset="car_rel_url_x", keep="last").copy()

In [60]:
exchange_rate = {"AZN": 1.0, "$": 1.70, "€": 1.85}
df_dedup["price_azn"] = df_dedup["price_x"] * df_dedup["currency_x"].map(exchange_rate)

In [61]:
exclude_body_types = ["Yük maşını", "Motosiklet", "Avtobus", "Moped", "Kvadrosikl", "Dartqı", "Mikroavtobus"]
df_clean = df_dedup[~df_dedup["Ban növü"].isin(exclude_body_types)].copy()
df_clean = df_clean[df_clean["price_azn"] >= 1000].copy()

In [62]:
print("Cleaned dataset shape:", df_clean.shape)

Cleaned dataset shape: (149478, 17)


## 2. Does gearbox type affect price?

Do automatic cars sell for a different average price than manual cars? This matters for a seller pricing a car and a buyer planning a budget.

### Variables
Dependent (continuous): price_azn

Independent (categorical, 2 groups): Sürətlər qutusu - "Mexaniki" (manual) vs "Avtomat" (automatic)


Continuous dependent variable + categorical independent variable with 2 groups independent samples t-test.

In [63]:
gearbox_preview = df_clean[df_clean["Sürətlər qutusu"].isin(["Mexaniki", "Avtomat"])].groupby("Sürətlər qutusu")["price_azn"].agg(["count", "mean", "median"])
gearbox_preview.round(0)


,count,mean,median
Sürətlər qutusu,,,
Avtomat,98161,30023.0,22000.0
Mexaniki,37161,10567.0,8900.0


Even if I find a significant difference, that doesn't mean choosing automatic makes the price go up. It's probably a confounding variable — automatic transmission shows up more in newer, more expensive cars, and those cars are already pricier on their own. So the link is probably indirect, not a direct cause.

## 3. Does brand affect price, and which brands differ?

This compares 5 groups, not 2. If I ran a separate t-test for every pair (5 brands = 10 pairs), the chance of getting at least one "significant" result just by luck goes way up, even with no real difference. This is the multiple comparisons problem.

Hypotheses: H0, H1

Continuous dependent variable + categorical independent variable with 5 groups one-way ANOVA, instead of many separate t-tests.

In [64]:
top5_brands = ["Mercedes", "Hyundai", "Kia", "Toyota", "LADA (VAZ)"]
brand_preview = df_clean[df_clean["Marka"].isin(top5_brands)].groupby("Marka")["price_azn"].agg(["count", "mean", "median"]).loc[top5_brands]
brand_preview.round(0)


,count,mean,median
Marka,,,
Mercedes,25307,25650.0,14300.0
Hyundai,19658,23864.0,22700.0
Kia,14619,25765.0,24500.0
Toyota,14388,30287.0,23200.0
LADA (VAZ),12859,7264.0,6100.0


In [65]:
from itertools import combinations
n_groups = len(top5_brands)
n_pairs = len(list(combinations(top5_brands, 2)))



In [66]:
print(f"Number of possible pairs for {n_groups} groups: {n_pairs}")
print(f"If I test each pair separately at 0.05, chance of at least one false positive: {1-(1-0.05)**n_pairs:.1%}")
print(f"(Bonferroni-corrected per test: {0.05/n_pairs:.4f})")

Number of possible pairs for 5 groups: 10
If I test each pair separately at 0.05, chance of at least one false positive: 40.1%
(Bonferroni-corrected per test: 0.0050)


If I ran all 10 comparisons separately at a=0.05 each, even with no real difference the chance of at least one false "significant" result reaches about 40%, too high. Bonferroni fixes this by lowering each test's α to 0.05/10 = 0.005.